# Hóemberek számának becslése

A Felső-AIföldi királyságban különösen nagy hagyománya van a hóember építésnek. Az általuk készített hóemberek nem különböznek különösebben a nálunk megtalálhatóaktól, viszont kitalált terület lévén komoly effortot tesznek a hóemberek számontartásába. Ennek köszönhetően hatalmas mennyiségű adatmennyiséget halmoztak fel a hóemberekhez kapcsolódó statisztikákból.

A Felső-AIföldi uralkodó arra kér, hogy a számukra oly fontos hóeberek kapcsán használd fel az eddig gyűjtött statisztikákat, és készíts becslést a hóemberek számára az esetlegesen kapcsolódó jellemzők alapján.

Tekintve, hogy a királyság igen nagy, így jelentős mennyiségű hóember található ott a tél egész folyamán.
Ebből kifolyólag igen nehéz pontos becslést adni a hóemberek tényleges számára. A pontosság mérésére így az átlagos négyzetes eltérés (MSE) helyett ennek a gyökét (RMSE) kell minimalizálni.

## Adatok

A kapcsolódó statisztikák az alábbiak (egy hétre véve):
- `temperature` - Átlagos heti hőmérséklet (float, °C)
- `wind_speed` - Szélsebesség (float, m/s)
- `wind_direction` - Északkal bezárt szélirány (float, radián)
- `precipation` - Csapadék átlagos napi mennyisége (float, mm)
- `freeze_days` - Fagyos napok száma a héten (int, azon napok száma, amikor az átlaghőmérséklet fagypont alatt volt)
- `carrot_price` - A sárgarépa kilónkénti ára (float, királyi egységes valuta/kg)
- `scarf_tag` - Volt-e a ruhaboltokban akció a sálra (bool, 0~nem, 1~igen)
- `action_movies` - A héten kijött akciófilmek száma (int, darabszám)
- `log_consumption` - Egy átlagos háztartás tűzifa-felhasználása (float, $m^3$)

**Célváltozó**
- `Hoember` (float) - A nyilvántartott hóemberek száma (csak a tanítási adatok esetén)

Emellett minden adat tartalmaz egy egyedi azonosító számot (`ID`), mely a teszteléshez fontos.

## Hiba számítása

Egy regressziós modell megadja a becsült kimeneti értéket a bemeneti feature-ök alapján. Példa egy kis adathalmazra:

| Minta | Valódi érték | Prediktált érték |
|-------|--------------|-------------------------|
| 1     | 1242       | 1235.043                    |
| 2     | 0.0            | 30.65                    |
| 3     | 324            | 566.74        |
| 4     | 0.0            | 42.43     |
| 5     | 2133.64            | 1799.21   |
| 6     | 642            | 1026.7   |

---

Az átlagos négyzetes eltérés gyökét az alábbi képlettel számoljuk:

$$\text{RMSE}(\hat{y}, y) = \sqrt{\frac{\sum_i^n (\hat{y}_i - y_i)^2}{n}}$$

Nézzük meg a fenti példákra a négyzetes eltéréseket:

| Minta | Valódi érték | Prediktált érték | Négyzetes eltérés |
|-------|--------------|-------------------------|----------|
| 1     | 1242       | 1235.043                    |   48.399849        |
| 2     | 0            | 30.65                    |   939.4225     |
| 3     | 324            | 566.74        |   58922.7076     |
| 4     | 0            | 42.43     | 1800.3049       |
| 5     | 2133            | 1799.21   |  111415.7641      |
| 6     | 642            | 1026.7   |  147225.69      |


Ezek átlaga: 

$$\frac{48.399849 + 939.4225 + 58922.7076 + 1800.3049 + 111415.7641 + 147225.69}{6} \\ \approx 53392.04815817$$

Vagyis az átlagos négyzetes eltérés gyöke:

$$\sqrt{53392.04815817} \approx 231.0672$$

Részletes leírás: [Link](https://en.wikipedia.org/wiki/Root_mean_square_deviation)

A következőkben betöltjük az adatokat, felosztjuk őket tanító és teszt adathalmazra, és egy egyszerű modellt "betanítunk" rajtuk.

Az ezen modell által visszaadott eredménynél kell **kisebb** RMSE értéket elérni.

## Importok

In [45]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.dummy import DummyRegressor

## Adat betöltés

In [46]:
labeled_df = pd.read_csv("train_labeled.csv")
labeled_features = labeled_df[labeled_df.columns[1:-1]].to_numpy()
labeled_out = labeled_df[labeled_df.columns[-1]].to_numpy()

test_df = pd.read_csv("test.csv")
test_features = test_df[test_df.columns[1:]].to_numpy()

print(labeled_features.shape, labeled_out.shape)
print(test_features.shape)

(1000, 9) (1000,)
(1000, 9)


In [47]:
labeled_df.describe()

,temperature,wind_speed,wind_direction,precipation,freeze_days,carrot_price,scarf_tag,action_movies,log_consumption,Hoember
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-4.724559,9.926591,3.128444,17.572980,4.025000,199.756640,0.493000,2.399000,49.649827,59729.836000
std,14.560306,5.753930,1.816457,13.384211,1.974405,29.074566,0.500201,1.720686,23.594678,69204.329237
min,-29.768399,0.002694,0.000073,0.027385,1.000000,92.538122,0.000000,0.000000,10.108442,0.000000
25%,-17.938598,5.182556,1.538522,6.239041,2.000000,180.229321,0.000000,1.000000,29.258930,0.000000
50%,-4.165223,9.960682,3.144618,14.628117,4.000000,199.816577,0.000000,2.000000,49.218028,15309.000000
75%,7.912233,14.865868,4.714850,25.956475,6.000000,218.915476,1.000000,4.000000,70.730257,133807.250000
max,19.970686,19.966950,6.280406,65.319592,7.000000,290.441351,1.000000,5.000000,89.976611,212396.000000


## Train-eval adathalmaz szétválasztás

In [48]:
train_x, eval_x, train_y, eval_y = train_test_split(
    labeled_features, labeled_out, test_size=0.2, random_state=43
)


## Modell betanítása

In [49]:
model = DummyRegressor()
model.fit(train_x, train_y)

print(root_mean_squared_error(eval_y, model.predict(eval_x)))


66228.86166921744


## Teszt kiértékelése és lementése

In [50]:
test_out = model.predict(test_features)

pd.DataFrame({"ID": test_df["ID"], "Hoember": test_out}).to_csv(
    "prediction.csv", index=False
)